In [ ]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")

# Load category mapping
cat_df = pd.read_excel(FILE_PATH, sheet_name="Sheet1")
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))

# Load master data
master = pd.read_excel(FILE_PATH, sheet_name="Master Data ")

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time", "Tonnage"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

# Aggregate & filter 120T only
records = []
for child, g in master.groupby("Child Part"):
    # Only if has at least one 120T machine
    if not (g["Tonnage"] == 120).any():
        continue

    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    machines = list(set(normalize_machine(x) for x in g["Vertical Machines"].dropna().unique() if normalize_machine(x)))
    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown"),
        "Inventory": g["Inventory_25"].iloc[0],
        "Tonnage_120": 1  # flag
    })

df_ml = pd.DataFrame(records)
print(f"Dataset ready for ML: {len(df_ml):,} parts on 120T machines")
print(df_ml["Category"].value_counts())

# Save to CSV for easy reuse
df_ml.to_csv("120t_ml_dataset.csv", index=False)
print("Saved → 120t_ml_dataset.csv")